# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [2]:
import pandas as pd
import scipy.stats as st
import numpy as np

# Load the Pokémon dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv"
df = pd.read_csv(url)

df.head()

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False


- We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

In [3]:
dragon_hp = df.loc[df["Type 1"] == "Dragon", "HP"].dropna()
grass_hp = df.loc[df["Type 1"] == "Grass", "HP"].dropna()

print("Dragon mean HP:", dragon_hp.mean())
print("Grass mean HP:", grass_hp.mean())

t_stat, p_value = st.ttest_ind(
    dragon_hp,
    grass_hp,
    alternative="greater",
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

Dragon mean HP: 83.3125
Grass mean HP: 67.27142857142857
T-statistic: 3.3349632905124063
P-value: 0.0007993609745420599


In [4]:
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

df.groupby("Legendary")[stats].mean().T

Legendary,False,True
HP,67.182313,92.738462
Attack,75.669388,116.676923
Defense,71.559184,99.661538
Sp. Atk,68.454422,122.184615
Sp. Def,68.892517,105.938462
Speed,65.455782,100.184615


- We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.


In [8]:
from statsmodels.stats.multitest import multipletests

stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

legendary = df[df["Legendary"] == True]
non_legendary = df[df["Legendary"] == False]

results = []

for stat in stats:
    t_stat, p_value = st.ttest_ind(
        legendary[stat].dropna(),
        non_legendary[stat].dropna(),
        equal_var=False,
        alternative="two-sided"
    )

    results.append({
        "Stat": stat,
        "Legendary mean": legendary[stat].mean(),
        "Non-Legendary mean": non_legendary[stat].mean(),
        "T-statistic": t_stat,
        "P-value": p_value
    })

results_df = pd.DataFrame(results)

results_df["Adjusted p-value"] = multipletests(
    results_df["P-value"],
    alpha=0.05,
    method="bonferroni"
)[1]

results_df["Significant"] = results_df["Adjusted p-value"] < 0.05

results_df

,Stat,Legendary mean,Non-Legendary mean,T-statistic,P-value,Adjusted p-value,Significant
0,HP,92.738462,67.182313,8.981370,1.002691e-13,6.016147e-13,True
1,Attack,116.676923,75.669388,10.438134,2.520372e-16,1.512223e-15,True
2,Defense,99.661538,71.559184,7.637078,4.826998e-11,2.896199e-10,True
3,Sp. Atk,122.184615,68.454422,13.417450,1.551461e-21,9.308768e-21,True
4,Sp. Def,105.938462,68.892517,10.015697,2.294933e-15,1.376960e-14,True
5,Speed,100.184615,65.455782,11.475044,1.049016e-18,6.294098e-18,True


### Challenge 2: Legendary vs. Non-Legendary Pokémon Stats

We reject the null hypothesis for all six statistics. Legendary Pokémon have statistically significantly different mean values for all six stats, and the observed means are higher than those of Non-Legendary Pokémon in every case.

**Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [5]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


**We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

In [15]:
import pandas as pd

df = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv"
)

print(df.columns.tolist())

['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


In [17]:
import numpy as np

school_lon, school_lat = -118, 34
hospital_lon, hospital_lat = -122, 37

def euclidean_distance(lon, lat, target_lon, target_lat):
    return np.sqrt(
        (lon - target_lon)**2 +
        (lat - target_lat)**2
    )

df["school_distance"] = euclidean_distance(
    df["longitude"], df["latitude"],
    school_lon, school_lat
)

df["hospital_distance"] = euclidean_distance(
    df["longitude"], df["latitude"],
    hospital_lon, hospital_lat
)

df[["school_distance", "hospital_distance"]].head()

,school_distance,hospital_distance
0,3.694888,8.187319
1,3.552591,7.966235
2,3.453940,8.143077
3,3.448840,8.154416
4,3.456848,8.183508


In [19]:
df["proximity"] = np.where(
    (df["school_distance"] < 0.50) |
    (df["hospital_distance"] < 0.50),
    "close",
    "far"
)

df["proximity"].value_counts()

proximity
far      10171
close     6829
Name: count, dtype: int64

In [20]:
df.groupby("proximity")["median_house_value"].agg(
    ["count", "mean", "median", "std"]
)

,count,mean,median,std
proximity,,,,
close,6829,246951.982135,218200.0,111823.983975
far,10171,180678.441058,149800.0,111019.113084


In [22]:
from scipy.stats import ttest_ind

close_prices = df.loc[
    df["proximity"] == "close", "median_house_value"
]

far_prices = df.loc[
    df["proximity"] == "far", "median_house_value"
]

t_stat, p_value = ttest_ind(
    close_prices,
    far_prices,
    equal_var=False,
    alternative="greater"
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 37.992330214201516
P-value: 1.5032478884296307e-301
